# Entity NER Agenda Graph — Gephi Export (Stage 2)

Reads back the curated review sheet and builds bipartite `nodes.csv` / `edges.csv`.
Structure mirrors the meta-topic bipartite graph: source sub-units ↔ entities.

**Source columns used:**
| Arena | File | Sub-unit col |
|---|---|---|
| news | `news/analysis/df_with_NER.csv` | `outlet` |
| talkshows | `subtitles/analysis/subs_with_NER.csv` | `program` |
| kamer | `tweede_kamer/analysis/Tweede_Kamer_with_NER.csv` | `type` |

**Edge weight** = fraction of sub-unit documents mentioning the entity (coverage share).  
Unlike the meta-topic graph, weights do NOT sum to 1 per sub-unit — a document mentions many entities.

In [1]:
import ast
import re
import pathlib
from collections import Counter
import pandas as pd

NB_DIR = pathlib.Path(".").resolve()

# ============================================================
# PARAMETERS — edit before running
# ============================================================

REVIEW_XLSX = "entity_review_edited.xlsx"

SOURCES = {
    "news": {
        "path":        "../../news/analysis/df_with_NER.csv",
        "subunit_col": "outlet",
        "node_type":   "news",
        "id_prefix":   "src",
    },
    "talkshows": {
        "path":        "../../subtitles/analysis/subs_with_NER.csv",
        "subunit_col": "program",
        "node_type":   "talkshows",
        "id_prefix":   "src",
    },
    "kamer": {
        "path":        "../../tweede_kamer/analysis/Tweede_Kamer_with_NER.csv",
        "subunit_col": "type",
        "node_type":   "kamer",
        "id_prefix":   "src",
    },
}

ENTITY_COLS = ["persons", "orgs", "countries"]   # order doesn't matter; all are parsed

OUT_NODES       = "entity_nodes.csv"
OUT_EDGES       = "entity_edges.csv"
MIN_DOCS        = 0    # drop sub-units with fewer total documents
MIN_EDGE_WEIGHT = 0.0  # drop edges below this coverage share
# ============================================================

## 1. Helpers (reused from Stage 1)

In [2]:
def parse_entity_list(cell):
    """Parse a list-valued cell → list of raw entity strings."""
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if s in ("", "[]", "nan"):
        return []
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [str(x) for x in result]
        return [str(result)]
    except (ValueError, SyntaxError):
        s = re.sub(r"^[\[\(]|[\]\)]$", "", s)
        return [x.strip().strip("'\"" ) for x in s.split(",") if x.strip()]


def normalise(s):
    """Collapse internal whitespace and strip edges."""
    return re.sub(r"\s+", " ", str(s).strip())


def slugify(s):
    """Lowercase alphanumeric slug; spaces/punctuation → underscore."""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")

## 2. Build entity lookup from review sheet

- `entity_to_canonical`: normalised raw entity (lowercase) → canonical display label  
- `canonical_to_type`: canonical label → PER / ORG / LOC  
  Uses the type from the row where `entity == canonical`; falls back to majority vote
  among merged members if no such row exists.

In [3]:
review_path = (NB_DIR / REVIEW_XLSX).resolve()
review_raw  = pd.read_excel(review_path)
review      = review_raw[review_raw["keep"].astype(str).str.strip().str.lower() == "yes"].copy()

print(f"Review sheet: {len(review_raw)} rows total, {len(review)} keep=yes")

# normalised entity string (lowercase) → canonical display label (case-preserved)
entity_to_canonical = {
    normalise(row["entity"]).lower(): normalise(row["canonical"])
    for _, row in review.iterrows()
}

# canonical display label → entity type
# Pass 1: rows where the entity IS the canonical (direct type)
canonical_to_type = {}
for _, row in review.iterrows():
    entity_key   = normalise(row["entity"]).lower()
    canonical    = normalise(row["canonical"])
    canonical_key = canonical.lower()
    if entity_key == canonical_key:
        canonical_to_type[canonical] = str(row["type"]).strip()

# Pass 2: canonicals not yet resolved → majority vote from merged members
canon_votes: dict[str, list] = {}
for _, row in review.iterrows():
    canonical = normalise(row["canonical"])
    if canonical not in canonical_to_type:
        canon_votes.setdefault(canonical, []).append(str(row["type"]).strip())

for canonical, types in canon_votes.items():
    canonical_to_type[canonical] = Counter(types).most_common(1)[0][0]

# Distinct canonicals (what will become entity nodes)
all_canonicals = set(entity_to_canonical.values())
print(f"Distinct canonical entities: {len(all_canonicals)}")
print(f"Alias mappings (entity != canonical): "
      f"{sum(1 for e, c in entity_to_canonical.items() if e != c.lower())}")

Review sheet: 150 rows total, 137 keep=yes
Distinct canonical entities: 123
Alias mappings (entity != canonical): 14


## 3. Load source files

In [4]:
frames = {}

for arena, cfg in SOURCES.items():
    abs_path = (NB_DIR / cfg["path"]).resolve()
    if not abs_path.exists():
        print(f"[SKIP] {arena}: not found at {abs_path}")
        continue
    df = pd.read_csv(abs_path)
    missing_ent = [c for c in ENTITY_COLS if c not in df.columns]
    if missing_ent:
        print(f"[SKIP] {arena}: missing entity columns {missing_ent}")
        continue
    if cfg["subunit_col"] not in df.columns:
        print(f"[SKIP] {arena}: missing sub-unit column '{cfg['subunit_col']}'")
        continue
    df[cfg["subunit_col"]] = df[cfg["subunit_col"]].fillna("UNKNOWN").astype(str).str.strip()
    frames[arena] = df
    print(
        f"[OK] {arena}: {len(df):,} docs, "
        f"{df[cfg['subunit_col']].nunique()} sub-units "
        f"(col: '{cfg['subunit_col']}')"
    )

[OK] news: 13,209 docs, 8 sub-units (col: 'outlet')
[OK] talkshows: 495 docs, 9 sub-units (col: 'program')
[OK] kamer: 844 docs, 3 sub-units (col: 'type')


## 4. Process documents → (sub-unit, canonical entity) pairs

For each document: parse all entity columns, normalise, map to canonical, dedupe
(each canonical counts at most once per document regardless of mention count),
drop anything not in the keep=yes lookup.

In [5]:
records    = []   # (arena, subunit, doc_id, canonical)
dropped_ct = 0

for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    df          = frames[arena]
    subunit_col = cfg["subunit_col"]

    for row_idx, row in df.iterrows():
        subunit = row[subunit_col]
        doc_id  = f"{arena}:{row_idx}"

        # Collect unique canonicals for this document
        doc_canonicals: set[str] = set()
        for col in ENTITY_COLS:
            for raw in parse_entity_list(row.get(col)):
                norm_key  = normalise(raw).lower()
                canonical = entity_to_canonical.get(norm_key)
                if canonical:
                    doc_canonicals.add(canonical)
                else:
                    dropped_ct += 1

        for canonical in doc_canonicals:
            records.append({
                "arena":     arena,
                "subunit":   subunit,
                "doc_id":    doc_id,
                "canonical": canonical,
            })

long_df = pd.DataFrame(records)
print(f"(sub-unit, canonical) pairings: {len(long_df):,}")
print(f"Entity mentions dropped (not in keep=yes lookup): {dropped_ct:,}")
print(long_df.groupby("arena").size().rename("pairings").to_string())

(sub-unit, canonical) pairings: 48,000
Entity mentions dropped (not in keep=yes lookup): 185,232
arena
kamer         2979
news         43791
talkshows     1230


## 5. Build nodes table

In [6]:
subunit_rows = []

for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    for subunit, grp in frames[arena].groupby(cfg["subunit_col"]):
        if len(grp) < MIN_DOCS:
            continue
        node_id = f"src_{slugify(subunit)}"
        subunit_rows.append({
            "Id":          node_id,
            "Label":       subunit,
            "node_type":   cfg["node_type"],
            "entity_type": "",
            "size":        len(grp),
        })

# Entity size = distinct documents across ALL sources mentioning this canonical
entity_doc_freq = (
    long_df.groupby("canonical")["doc_id"]
    .nunique()
    .rename("size")
)

entity_rows = [
    {
        "Id":          f"ent_{slugify(canonical)}",
        "Label":       canonical,
        "node_type":   "entity",
        "entity_type": canonical_to_type.get(canonical, ""),
        "size":        int(entity_doc_freq.get(canonical, 0)),
    }
    for canonical in sorted(all_canonicals)
    if canonical in entity_doc_freq  # drop canonicals that appear in no kept document
]

nodes_df = pd.DataFrame(subunit_rows + entity_rows)

# Collision check
dupes = nodes_df[nodes_df.duplicated("Id", keep=False)]
if len(dupes):
    print("WARNING — duplicate node Ids:", dupes["Id"].tolist())
else:
    print("Node Ids: no collisions ✓")

print(f"Total nodes: {len(nodes_df)}  "
      f"({len(subunit_rows)} sub-units + {len(entity_rows)} entities)")
nodes_df

Node Ids: no collisions ✓
Total nodes: 143  (20 sub-units + 123 entities)


,Id,Label,node_type,entity_type,size
0,src_ad,AD,news,,3595
1,src_ga,GA,news,,401
2,src_nrc,NRC,news,,2071
3,src_nu_nl,NU.nl,news,,1076
4,src_parool,Parool,news,,448
...,...,...,...,...,...
138,ent_whatsapp,WhatsApp,entity,ORG,176
139,ent_witte_huis,Witte Huis,entity,ORG,117
140,ent_zuid_korea,Zuid-Korea,entity,LOC,155
141,ent_zweden,Zweden,entity,LOC,183


## 6. Build edges table

**Weight** = distinct documents in sub-unit mentioning this entity / total documents in sub-unit.  
This is a coverage share — weights do NOT sum to 1 per sub-unit.

In [7]:
# Sub-unit total document counts (denominator)
subunit_totals = {}
for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    for subunit, grp in frames[arena].groupby(cfg["subunit_col"]):
        if len(grp) >= MIN_DOCS:
            subunit_totals[(arena, subunit)] = len(grp)

# Distinct-document counts per (arena, subunit, canonical)
mention_counts = (
    long_df.groupby(["arena", "subunit", "canonical"])["doc_id"]
    .nunique()
    .reset_index(name="mention_docs")
)

# Node Id lookups
subunit_id = {
    row["Label"]: row["Id"]
    for _, row in nodes_df[nodes_df["node_type"] != "entity"].iterrows()
}
entity_id = {
    row["Label"]: row["Id"]
    for _, row in nodes_df[nodes_df["node_type"] == "entity"].iterrows()
}

edge_rows = []
for _, row in mention_counts.iterrows():
    arena    = row["arena"]
    subunit  = row["subunit"]
    canonical = row["canonical"]
    key = (arena, subunit)

    if key not in subunit_totals:
        continue   # sub-unit was dropped by MIN_DOCS
    src_id = subunit_id.get(subunit)
    tgt_id = entity_id.get(canonical)
    if src_id is None or tgt_id is None:
        continue

    weight = row["mention_docs"] / subunit_totals[key]
    if weight < MIN_EDGE_WEIGHT:
        continue

    edge_rows.append({
        "Source": src_id,
        "Target": tgt_id,
        "Weight": round(weight, 6),
        "Type":   "Undirected",
    })

edges_df = pd.DataFrame(edge_rows)
print(f"Total edges: {len(edges_df)}")
edges_df

Total edges: 1473


,Source,Target,Weight,Type
0,src_beleidsnota,ent_ad,0.002915,Undirected
1,src_beleidsnota,ent_asml,0.014577,Undirected
2,src_beleidsnota,ent_afrika,0.002915,Undirected
3,src_beleidsnota,ent_amazon,0.005831,Undirected
4,src_beleidsnota,ent_amsterdam,0.008746,Undirected
...,...,...,...,...
1468,src_pauw,ent_new_york,0.052632,Undirected
1469,src_pauw,ent_pvv,0.052632,Undirected
1470,src_pauw,ent_verenigde_staten,0.105263,Undirected
1471,src_pauw,ent_zuid_korea,0.052632,Undirected


## 7. Diagnostic summary

In [8]:
print("=" * 60)
print("DIAGNOSTIC SUMMARY")
print()

print("SUB-UNIT COUNTS PER ARENA:")
for arena, cfg in SOURCES.items():
    arena_nodes = nodes_df[nodes_df["node_type"] == cfg["node_type"]]
    if arena not in frames:
        print(f"  {arena}: [skipped]")
    else:
        print(f"  {arena}: {len(arena_nodes)} sub-unit nodes")
        for _, row in arena_nodes.sort_values("size", ascending=False).iterrows():
            print(f"      {row['Label']:<35s}  {row['size']:>6,} docs")
ent_nodes = nodes_df[nodes_df["node_type"] == "entity"]
print(f"  entity: {len(ent_nodes)} nodes")
print()

print(f"TOTAL EDGES: {len(edges_df)}")
print()

print("TOP 10 ENTITIES BY SIZE (distinct-document frequency):")
top10 = (
    ent_nodes
    .sort_values("size", ascending=False)
    .head(10)[["Label", "entity_type", "size"]]
)
print(top10.to_string(index=False))
print()

print("ENTITY TYPE BREAKDOWN:")
print(ent_nodes["entity_type"].value_counts().to_string())
print()

print("COVERAGE SHARE STATISTICS (edge weights):")
print(edges_df["Weight"].describe().round(4).to_string())
print("=" * 60)

DIAGNOSTIC SUMMARY

SUB-UNIT COUNTS PER ARENA:
  news: 8 sub-unit nodes
      AD                                    3,595 docs
      VK                                    2,423 docs
      TG                                    2,297 docs
      NRC                                   2,071 docs
      NU.nl                                 1,076 docs
      TR                                      898 docs
      Parool                                  448 docs
      GA                                      401 docs
  talkshows: 9 sub-unit nodes
      goedemorgen nederland                   296 docs
      de wereld draait door                    70 docs
      eva                                      42 docs
      jinek                                    23 docs
      bar laat                                 21 docs
      pauw                                     19 docs
      café kockelmann                          14 docs
      de vooravond                              6 docs
      m           

## 8. Export

In [9]:
nodes_df.to_csv(OUT_NODES, index=False)
edges_df.to_csv(OUT_EDGES, index=False)

print(f"Saved {len(nodes_df)} nodes  -> {OUT_NODES}")
print(f"Saved {len(edges_df)} edges  -> {OUT_EDGES}")
print()
print("Node type breakdown:")
print(nodes_df["node_type"].value_counts().to_string())

Saved 143 nodes  -> entity_nodes.csv
Saved 1473 edges  -> entity_edges.csv

Node type breakdown:
node_type
entity       123
talkshows      9
news           8
kamer          3
